# TF-IDF + Logistic Regression Baseline
Sanity-check baseline for the PEFT vs FFT IndoBERT fake news study. 

**What this answers**
- Section 1-5: how well does a bag-of-words model do on the same split? (if close to IndoBERT, the task is near ceiling)
- Section 6: how does it scale with data size (5/10/50/100%) and seeds?
- Section 7: which tokens drive predictions, and how detectable is source style?
- Section 8: Summary

## Section 1: Imports and config

In [ ]:
import numpy as np, pandas as pd, time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix

DATA_PATH = 'datasets\cleaned\combined_news_cleaned.csv'
TEXT_COL = 'berita'
LABEL_COL = 'label'
HOAX_VALUE = 1        # value in label that means hoax (e.g. 'hoax' or 1)
SEEDS = [42, 43, 44, 45, 46]
SIZES = [0.05, 0.10, 0.50, 1.00]
SPLIT = (0.70, 0.10, 0.20)   # train / val / test

summary = []
print('Config loaded. Seeds:', SEEDS, '| sizes:', SIZES, '| split:', SPLIT)

<>:7: SyntaxWarning: invalid escape sequence '\c'
<>:7: SyntaxWarning: invalid escape sequence '\c'
C:\Users\screm\AppData\Local\Temp\ipykernel_7352\2095351867.py:7: SyntaxWarning: invalid escape sequence '\c'
  DATA_PATH = 'datasets\cleaned\combined_news_cleaned.csv'


Config loaded. Seeds: [42, 43, 44, 45, 46] | sizes: [0.05, 0.1, 0.5, 1.0] | split: (0.7, 0.1, 0.2)


## Section 2: Load data and inspect

In [2]:
df = pd.read_csv(DATA_PATH, usecols=[TEXT_COL, LABEL_COL])   # no source/date loaded
df = df.dropna().reset_index(drop=True)
df['y'] = (df[LABEL_COL] == HOAX_VALUE).astype(int)
texts = df[TEXT_COL].astype(str).values
y = df['y'].to_numpy(dtype=int)
print('Raw label values:')
print(df[LABEL_COL].value_counts(dropna=False).to_string())
assert set(df['y'].unique()) == {0, 1}, 'y must be binary 0/1, check HOAX_VALUE and the label column'

print('Rows:', len(df))
print('Label counts (1 = hoax):')
print(df['y'].value_counts().to_string())
print('Median words per article:', df[TEXT_COL].astype(str).str.split().str.len().median())

Raw label values:
label
0    6062
1    6062
Rows: 12124
Label counts (1 = hoax):
y
0    6062
1    6062
Median words per article: 10.0


## Section 3: Train / val / test split (stratified by label, source, month)
To compare fairly with IndoBERT, replace the indices below with your saved split indices (tr_idx, va_idx, te_idx).

In [3]:
idx = np.arange(len(df))
trval_idx, te_idx = train_test_split(idx, test_size=SPLIT[2], stratify=y, random_state=42)
tr_idx, va_idx = train_test_split(trval_idx, test_size=SPLIT[1] / (SPLIT[0] + SPLIT[1]),
                                  stratify=y[trval_idx], random_state=42)

for name, ix in [('train', tr_idx), ('val', va_idx), ('test', te_idx)]:
    classes, counts = np.unique(y[ix], return_counts=True)
    print(name, 'n =', len(ix), '| label counts =', dict(zip(classes.tolist(), counts.tolist())),
          '| hoax share =', round(y[ix].mean(), 3))
    assert len(classes) == 2, f'{name} split must contain both classes; got {classes.tolist()}'
print('Overlap check (should be 0):', len(set(tr_idx) & set(te_idx)), len(set(va_idx) & set(te_idx)))

train n = 8486 | label counts = {0: 4243, 1: 4243} | hoax share = 0.5
val n = 1213 | label counts = {0: 606, 1: 607} | hoax share = 0.5
test n = 2425 | label counts = {0: 1213, 1: 1212} | hoax share = 0.5
Overlap check (should be 0): 0 0


## Section 4: Helper functions

In [4]:
def evaluate(model, X, y_true):
    p = model.predict(X)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, p, average='macro', zero_division=0)
    return dict(acc=accuracy_score(y_true, p), prec=pr, rec=rc, f1=f1)

def fit_tfidf_lr(tr_texts, tr_y, va_texts, va_y, analyzer='word', ngram=(1, 2),
                 Cs=(0.1, 1, 10, 100), max_features=None):
    vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, min_df=2, max_df=0.95,
                          sublinear_tf=True, lowercase=True, max_features=max_features)
    Xtr = vec.fit_transform(tr_texts)
    Xva = vec.transform(va_texts)
    best = None
    for C in Cs:
        clf = LogisticRegression(C=C, max_iter=2000, solver='lbfgs').fit(Xtr, tr_y)
        f1 = f1_score(va_y, clf.predict(Xva), average='macro')
        if best is None or f1 > best[0]:
            best = (f1, C, clf)
    return vec, best[2], best[1], best[0]

print('Helpers defined: evaluate(), fit_tfidf_lr()')

Helpers defined: evaluate(), fit_tfidf_lr()


## Section 5: Main baseline (word 1-2 grams, 100% train data)
C is tuned on the validation set only; the test set is touched once.

In [5]:
t0 = time.time()
vec_w, clf_w, best_C, val_f1 = fit_tfidf_lr(texts[tr_idx], y[tr_idx], texts[va_idx], y[va_idx])
print('Best C (by val macro-F1):', best_C, '| val F1:', round(val_f1, 4))

Xte = vec_w.transform(texts[te_idx])
m = evaluate(clf_w, Xte, y[te_idx])
print('Test metrics:', {k: round(v, 4) for k, v in m.items()})
print('Confusion matrix (rows = true fact/hoax, cols = predicted fact/hoax):')
print(confusion_matrix(y[te_idx], clf_w.predict(Xte)))
print('Vocabulary size:', len(vec_w.vocabulary_), '| time:', round(time.time() - t0, 1), 's')
summary.append(dict(setting='word 1-2gram, random split, 100%', **m))

Best C (by val macro-F1): 10 | val F1: 0.8714
Test metrics: {'acc': 0.8812, 'prec': 0.8813, 'rec': 0.8812, 'f1': 0.8812}
Confusion matrix (rows = true fact/hoax, cols = predicted fact/hoax):
[[1076  137]
 [ 151 1061]]
Vocabulary size: 13868 | time: 0.7 s


## Section 5b: Character n-gram variant (3-5, char_wb)

In [6]:
t0 = time.time()
vec_c, clf_c, C_c, vf_c = fit_tfidf_lr(texts[tr_idx], y[tr_idx], texts[va_idx], y[va_idx],
                                       analyzer='char_wb', ngram=(3, 5), Cs=(1, 10, 100), max_features=300000)
m_c = evaluate(clf_c, vec_c.transform(texts[te_idx]), y[te_idx])
print('Best C:', C_c, '| val F1:', round(vf_c, 4))
print('Test metrics:', {k: round(v, 4) for k, v in m_c.items()})
print('time:', round(time.time() - t0, 1), 's')
summary.append(dict(setting='char_wb 3-5gram, random split, 100%', **m_c))

Best C: 10 | val F1: 0.864
Test metrics: {'acc': 0.8891, 'prec': 0.8891, 'rec': 0.8891, 'f1': 0.8891}
time: 3.3 s


## Section 6: Data-size x seed curve (5 / 10 / 50 / 100%)
Subsets are stratified by label and source and drawn per seed from the training split. The val and test sets are fixed. At 100% every seed trains on the same data, so the std is about 0 (liblinear is near-deterministic).

In [7]:
rows = []
for frac in SIZES:
    res = []
    for seed in SEEDS:
        if frac < 1.0:
            sub, _ = train_test_split(tr_idx, train_size=frac, stratify=y[tr_idx], random_state=seed)
        else:
            sub = tr_idx
        vec, clf, C, _ = fit_tfidf_lr(texts[sub], y[sub], texts[va_idx], y[va_idx])
        res.append(evaluate(clf, vec.transform(texts[te_idx]), y[te_idx]))
    r = pd.DataFrame(res)
    print('--- size', int(frac * 100), '% | n_train =', len(sub), '---')
    print((r.mean().round(4).astype(str) + ' +/- ' + r.std().round(4).astype(str)).to_string())
    rows.append(dict(size=frac, n_train=len(sub), f1_mean=r.f1.mean(), f1_std=r.f1.std(), acc_mean=r.acc.mean()))
    summary.append(dict(setting='word 1-2gram, ' + str(int(frac * 100)) + '% data (mean of ' + str(len(SEEDS)) + ' seeds)', **r.mean().to_dict()))
print()
print(pd.DataFrame(rows).round(4).to_string(index=False))

--- size 5 % | n_train = 424 ---
acc     0.7511 +/- 0.0071
prec    0.7519 +/- 0.0064
rec     0.7511 +/- 0.0071
f1      0.7509 +/- 0.0072
--- size 10 % | n_train = 848 ---
acc     0.7838 +/- 0.0075
prec    0.7855 +/- 0.0059
rec     0.7837 +/- 0.0075
f1      0.7834 +/- 0.0079
--- size 50 % | n_train = 4243 ---
acc     0.8576 +/- 0.0041
prec    0.8578 +/- 0.0041
rec     0.8576 +/- 0.0041
f1      0.8576 +/- 0.0042
--- size 100 % | n_train = 8486 ---
acc     0.8812 +/- 0.0
prec    0.8813 +/- 0.0
rec     0.8812 +/- 0.0
f1      0.8812 +/- 0.0

 size  n_train  f1_mean  f1_std  acc_mean
 0.05      424   0.7509  0.0072    0.7511
 0.10      848   0.7834  0.0079    0.7838
 0.50     4243   0.8576  0.0042    0.8576
 1.00     8486   0.8812  0.0000    0.8812


## Section 7: Top discriminative tokens (artifact check)
Look for source names, boilerplate, or debunking phrases (e.g. klarifikasi, faktanya, datelines) that would let the model cheat.

In [8]:
names = np.array(vec_w.get_feature_names_out())
coef = clf_w.coef_[0]
order = np.argsort(coef)
print('Top 30 tokens -> HOAX:')
print(', '.join(names[order[-30:]][::-1]))
print()
print('Top 30 tokens -> FACT:')
print(', '.join(names[order[:30]]))

Top 30 tokens -> HOAX:
tautan, foto, anies, gibran, video, karena, jokowi, lowongan, akun, uang, dokumentasi, pendaftaran, whatsapp, cina, purbaya, lowongan kerja, indonesia, artikel, vaksin, undian, bantuan, oleh, melalui, demo, rekrutmen, gambar, adalah, israel, melarang, tautan pendaftaran

Top 30 tokens -> FACT:
tak, rp, soal, momen, di, ini, diduga, apa, pastikan, ungkap, dugaan, kronologi, pasar, besok, dorong, begini, masih, hingga, tewas, vs, kita, bertemu, lebih, hukum, nasdem, cerita, cek, jangan, waspada, gelar


## Section 8: Summary table

In [9]:
print(pd.DataFrame(summary).round(4).to_string(index=False))
print()
print('How to read this:')
print('How to read this:')
print('- Test F1 close to IndoBERT (~0.92): task is near ceiling, PEFT gaps will be small.')
print('- Check the Block 7 tokens for source names or debunking phrases that would act as shortcuts.')

                                  setting    acc   prec    rec     f1
         word 1-2gram, random split, 100% 0.8812 0.8813 0.8812 0.8812
      char_wb 3-5gram, random split, 100% 0.8891 0.8891 0.8891 0.8891
  word 1-2gram, 5% data (mean of 5 seeds) 0.7511 0.7519 0.7511 0.7509
 word 1-2gram, 10% data (mean of 5 seeds) 0.7838 0.7855 0.7837 0.7834
 word 1-2gram, 50% data (mean of 5 seeds) 0.8576 0.8578 0.8576 0.8576
word 1-2gram, 100% data (mean of 5 seeds) 0.8812 0.8813 0.8812 0.8812

How to read this:
How to read this:
- Test F1 close to IndoBERT (~0.92): task is near ceiling, PEFT gaps will be small.
- Check the Block 7 tokens for source names or debunking phrases that would act as shortcuts.
